# Ad Click Prediction (ROC-AUC)

Predict whether an ad impression results in a click. Cells are split into imports, preprocessing, model fitting and submission.

Pick one of three model versions with `VARIANT` in the model fitting cell:

| VARIANT | Models | Logistic regression |
|---|---|---|
| `ensemble_l1` | RF, ET, LightGBM, XGBoost, LogReg | L1 (lasso, saga) |
| `ensemble_l2` | RF, ET, LightGBM, XGBoost, LogReg | L2 (ridge, lbfgs) |
| `lean_lgb_logreg` | LightGBM, LogReg | L2 (ridge, lbfgs) |

Data files are read from `../data/`. In Colab, upload the two CSVs and set `DATA_DIR = '.'`.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import lightgbm as lgb
import xgboost as xgb

print('Imports loaded')

In [ ]:
DATA_DIR = '../data'
SAMPLE = None  # set to a row count for a quick test run

train = pd.read_csv(f'{DATA_DIR}/QM571_Train.csv')
test = pd.read_csv(f'{DATA_DIR}/QM571_Test_X.csv')

# drop rows with a missing target ( clicked )
train = train.dropna(subset=['clicked']).reset_index(drop=True)
if SAMPLE:
    train = train.sample(n=SAMPLE, random_state=42).reset_index(drop=True)

y = train['clicked'].astype(int).values
ids = test['impression_id'].values
X = train.drop(columns=['clicked'])
T = test.drop(columns=['impression_id'])
if 'impression_id' in X.columns:
    X = X.drop(columns=['impression_id'])

neg1_cols = ['previous_publisher_tenure_days', 'current_publisher_tenure_days',
             'cookie_age_days', 'session_elapsed_minutes', 'device_distinct_users_8w']
for df in (X, T):
    # flag each -1 "unavailable" value, then set it to NaN
    for c in neg1_cols:
        df[c + '_na'] = (df[c] == -1).astype(int)
        df[c] = df[c].replace(-1, np.nan)

    # engineered ratio and interaction features
    df['user_publisher_tenure_ratio'] = (
        df['current_publisher_tenure_days'].fillna(0) /
        (df['previous_publisher_tenure_days'].fillna(1) + 1)
    )
    df['session_intensity'] = (
        df['session_elapsed_minutes'] * df['device_distinct_users_8w']
    ).fillna(0)
    df['cookie_session_ratio'] = (
        df['cookie_age_days'] / (df['session_elapsed_minutes'].fillna(1) + 1)
    )

# fill missing categoricals with the most frequent training value
cat_cols = ['ad_format_code', 'audience_segment_code', 'ad_placement_code',
            'traffic_channel', 'device_platform']
for c in cat_cols:
    mode = X[c].mode(dropna=True)[0]
    X[c] = X[c].fillna(mode)
    T[c] = T[c].fillna(mode)

# one-hot encode categoricals on train and test together so columns match
n = len(X)
full = pd.get_dummies(pd.concat([X, T], axis=0).astype({c: str for c in cat_cols}),
                      columns=cat_cols, dtype=int)
X_p, T_p = full.iloc[:n].reset_index(drop=True), full.iloc[n:].reset_index(drop=True)

# median-filled copy for Random Forest and Extra Trees
med = X_p.median()
X_f, T_f = X_p.fillna(med), T_p.fillna(med)

print('Train:', X_p.shape, '| Test:', T_p.shape, '| Click rate:', round(y.mean(), 4))

In [ ]:
VARIANT = 'ensemble_l1'  # 'ensemble_l1', 'ensemble_l2' or 'lean_lgb_logreg'

VARIANTS = {
    'ensemble_l1': {'names': ['rf', 'et', 'lgb', 'xgb', 'logreg'],
                    'logreg': {'penalty': 'l1', 'solver': 'saga'}},
    'ensemble_l2': {'names': ['rf', 'et', 'lgb', 'xgb', 'logreg'],
                    'logreg': {'penalty': 'l2', 'solver': 'lbfgs'}},
    'lean_lgb_logreg': {'names': ['lgb', 'logreg'],
                        'logreg': {'penalty': 'l2', 'solver': 'lbfgs'}},
}
names = VARIANTS[VARIANT]['names']
logreg_cfg = VARIANTS[VARIANT]['logreg']
uses_filled = {'rf', 'et'}

# model definitions; n_iter=None uses early stopping, n_iter=int fixes the number of trees
def make_model(name, n_iter=None):
    fixed = n_iter is not None
    if name == 'rf':
        return RandomForestClassifier(n_estimators=500, min_samples_leaf=20, max_features=0.4,
                                      n_jobs=-1, random_state=42)
    if name == 'et':
        return ExtraTreesClassifier(n_estimators=500, min_samples_leaf=20, max_features=0.5,
                                    n_jobs=-1, random_state=42)
    if name == 'lgb':
        return lgb.LGBMClassifier(n_estimators=n_iter or 3000, learning_rate=0.02, num_leaves=8,
                                  min_child_samples=100, subsample=0.7, subsample_freq=1,
                                  colsample_bytree=0.7, reg_lambda=5, random_state=42, verbose=-1)
    if name == 'xgb':
        return xgb.XGBClassifier(n_estimators=n_iter or 3000, learning_rate=0.02, max_depth=3,
                                 min_child_weight=10, subsample=0.7, colsample_bytree=0.7,
                                 reg_lambda=5, eval_metric='auc',
                                 early_stopping_rounds=None if fixed else 100,
                                 random_state=42, n_jobs=-1)
    if name == 'logreg':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('lr', LogisticRegressionCV(Cs=[0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10],
                                        scoring='roc_auc', cv=3, max_iter=5000, n_jobs=-1,
                                        **logreg_cfg)),
        ])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = {k: np.zeros(n) for k in names}
best_iters = {k: [] for k in ['lgb', 'xgb'] if k in names}

for fold, (tr, va) in enumerate(skf.split(X_p, y)):
    for name in names:
        Xa = X_f if name in uses_filled else X_p
        Xtr, Xva, ytr, yva = Xa.iloc[tr], Xa.iloc[va], y[tr], y[va]
        model = make_model(name)
        if name == 'lgb':
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric='auc',
                      callbacks=[lgb.early_stopping(100, verbose=False)])
            best_iters[name].append(model.best_iteration_)
        elif name == 'xgb':
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
            best_iters[name].append(model.best_iteration + 1)
        else:
            model.fit(Xtr, ytr)
        oof[name][va] = model.predict_proba(Xva)[:, 1]
    print(f'Fold {fold + 1} done')

# blend weights that maximize out-of-fold AUC on raw probabilities
O = np.column_stack([oof[k] for k in names])

def neg_auc(w):
    w = np.abs(w) / np.abs(w).sum()
    return -roc_auc_score(y, O @ w)

res = minimize(neg_auc, np.ones(len(names)) / len(names), method='Nelder-Mead',
               options={'maxiter': 2000})
weights = np.abs(res.x) / np.abs(res.x).sum()

print('\nPer-model results:')
for idx, k in enumerate(names):
    print(f'{k}: OOF AUC = {roc_auc_score(y, oof[k]):.5f}, Weight = {weights[idx]:.3f}')

print(f'\nEqual-weight ensemble OOF AUC: {roc_auc_score(y, O.mean(axis=1)):.5f}')
print(f'Weighted ensemble OOF AUC: {-res.fun:.5f}')
print('Avg best iterations:', {k: int(np.mean(v)) for k, v in best_iters.items()})

In [ ]:
# refit each model on all training data using the average best iteration from CV
test_preds = {}
for name in names:
    Xa, Ta = (X_f, T_f) if name in uses_filled else (X_p, T_p)
    n_iter = int(np.mean(best_iters[name]) * 1.1) if name in best_iters else None
    model = make_model(name, n_iter=n_iter)
    model.fit(Xa, y)
    test_preds[name] = model.predict_proba(Ta)[:, 1]
    print(f'{name} fitted on full train')

# weighted average of predicted probabilities
P = np.column_stack([test_preds[k] for k in names])
final_pred = P @ weights

submission = pd.DataFrame({'impression_id': ids, 'clicked': final_pred})
submission.to_csv(f'{VARIANT}_submission.csv', index=False)
print('Submission saved:', submission.shape)
print('Prediction range:', final_pred.min().round(4), '-', final_pred.max().round(4))
print(submission.head())